## 1. What Problem Do Streams and Tasks Solve?

In data pipelines, you often need to:
1. **Detect** which rows changed in a source table (inserts, updates, deletes).
2. **React** to those changes automatically — transform and load them downstream.

**Streams** solve problem #1 (change detection).  
**Tasks** solve problem #2 (scheduled/triggered execution).  
Together, they form Snowflake's native **CDC (Change Data Capture) + orchestration** layer — no external tools needed.

---
## 2. Streams — Change Data Capture

### What is a Stream?
A **stream** is a metadata object that records **DML changes** (INSERT, UPDATE, DELETE) made to a table (or view/external table) since the last time the stream was consumed.

It does **not** copy or store the data. It tracks an **offset** — a pointer to the table's change history.

### Key Mental Model

```
  Table (source)
     |
     | DML happens (INSERT/UPDATE/DELETE)
     v
  Stream (offset pointer)
     |
     | SELECT from stream shows only the NEW changes
     v
  Consumer (a DML statement inside a transaction)
```

When you successfully **consume** a stream (SELECT from it inside a DML that commits), the offset advances. The consumed changes disappear from the stream.

### Stream Columns

When you `SELECT * FROM my_stream`, you see all the source table's columns **plus** three metadata columns:

| Metadata Column | Type | Meaning |
|---|---|---|
| `METADATA$ACTION` | VARCHAR | `INSERT` or `DELETE` |
| `METADATA$ISUPDATE` | BOOLEAN | `TRUE` if the row is part of an UPDATE (UPDATEs show as DELETE + INSERT pair) |
| `METADATA$ROW_ID` | VARCHAR | Unique ID for the row across changes |

### How UPDATEs Appear
An UPDATE on a row produces **two rows** in the stream:
1. A `DELETE` row with the **old** values (`METADATA$ISUPDATE = TRUE`)
2. An `INSERT` row with the **new** values (`METADATA$ISUPDATE = TRUE`)

This lets you compare old vs. new values for the same row using `METADATA$ROW_ID`.

### Stream Types

| Type | Tracks | Use Case |
|---|---|---|
| **Standard** (default) | INSERTs, UPDATEs, DELETEs | Full CDC — you need to know everything |
| **Append-only** | Only INSERTs | Event/log tables where rows are never updated or deleted |
| **Insert-only** | Only INSERTs on external tables | External tables (which don't support UPDATE/DELETE) |

### Syntax

```sql
-- Standard stream
CREATE OR REPLACE STREAM my_stream ON TABLE my_table;

-- Append-only stream
CREATE OR REPLACE STREAM my_stream ON TABLE my_table APPEND_ONLY = TRUE;

-- Stream on a view
CREATE OR REPLACE STREAM my_stream ON VIEW my_view;
```

### Stream Staleness

A stream becomes **stale** when the source table's change history no longer covers the stream's offset. This happens when:
- The `DATA_RETENTION_TIME_IN_DAYS` on the source table is too short.
- The stream goes unconsumed for longer than the retention period.

**A stale stream cannot be read.** You must recreate it.

**Prevention:** Set `DATA_RETENTION_TIME_IN_DAYS` on the source table to be longer than the maximum gap between stream consumption.

```sql
ALTER TABLE my_table SET DATA_RETENTION_TIME_IN_DAYS = 14;
```

### Consuming a Stream (Offset Advance Rules)

The stream offset advances **only** when:
1. You SELECT from the stream inside a DML statement (INSERT, MERGE, etc.).
2. That DML is inside a transaction.
3. The transaction **commits** successfully.

```sql
-- This advances the offset:
INSERT INTO target_table
SELECT * FROM my_stream;  -- offset advances on commit

-- This does NOT advance the offset:
SELECT * FROM my_stream;  -- just reading, no DML wrapping it
```

If the transaction rolls back, the offset stays where it was — no data loss.

---
## 3. Tasks — Scheduled / Triggered Execution

### What is a Task?
A **task** is a Snowflake object that executes a **single SQL statement** on a schedule or in response to a trigger (like a stream having data).

Think of it as a lightweight cron job that lives inside Snowflake.

### Key Properties

| Property | Description |
|---|---|
| `SCHEDULE` | Cron expression or interval (e.g., `'1 MINUTE'`, `'USING CRON 0 9 * * * UTC'`) |
| `WAREHOUSE` | Which warehouse executes the SQL (or use serverless tasks) |
| `WHEN` | A boolean condition — task runs only if this is TRUE |
| `AS` | The SQL statement to execute |

### Basic Task Syntax

```sql
CREATE OR REPLACE TASK my_task
  WAREHOUSE = 'COMPUTE_WH'
  SCHEDULE = '5 MINUTE'             -- run every 5 minutes
  WHEN SYSTEM$STREAM_HAS_DATA('my_stream')  -- only if stream has data
AS
  INSERT INTO target_table
  SELECT col1, col2
  FROM my_stream;
```

### Critical: Tasks are Created SUSPENDED

Every new task starts in a **suspended** state. You must explicitly resume it:

```sql
ALTER TASK my_task RESUME;
```

To pause it later:
```sql
ALTER TASK my_task SUSPEND;
```

### WHEN Condition + SYSTEM$STREAM_HAS_DATA

The `WHEN` clause prevents the task from executing (and consuming warehouse credits) when there's nothing to do.

```sql
WHEN SYSTEM$STREAM_HAS_DATA('my_db.my_schema.my_stream')
```

- Returns `TRUE` if the stream has unconsumed changes.
- Returns `FALSE` otherwise — the task skips that scheduled run.
- The check itself is **free** (no warehouse needed).

Without `WHEN`, the task runs on every scheduled interval regardless, which wastes credits if there's no new data.

### Task Trees (DAGs)

Tasks can be chained into a **Directed Acyclic Graph (DAG)**:

```
  Root Task (has SCHEDULE)
     |
     +---> Child Task A (AFTER root_task)
     |         |
     |         +---> Grandchild Task C (AFTER child_task_a)
     |
     +---> Child Task B (AFTER root_task)
```

**Rules:**
- Only the **root task** has a `SCHEDULE`.
- Child tasks use `AFTER parent_task` instead of a schedule.
- A child runs only after its parent completes successfully.
- All tasks in a DAG share the same warehouse (unless overridden).

```sql
-- Root
CREATE TASK root_task
  WAREHOUSE = 'COMPUTE_WH'
  SCHEDULE = '10 MINUTE'
AS ...;

-- Child (runs after root completes)
CREATE TASK child_task
  WAREHOUSE = 'COMPUTE_WH'
  AFTER root_task
AS ...;
```

**Resuming a DAG:** You must resume tasks bottom-up (children first, then root). Suspending is top-down (root first).

### Serverless Tasks

Instead of specifying a `WAREHOUSE`, you can let Snowflake manage compute:

```sql
CREATE TASK my_task
  USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
  SCHEDULE = '5 MINUTE'
AS ...;
```

- Snowflake auto-scales compute.
- You pay only for actual execution time.
- No warehouse to manage.
- Good for intermittent/short workloads.

---
## 4. Streams + Tasks Together — The Pattern

The canonical CDC pipeline pattern:

```
 Source Table
     |
     | (DML changes happen)
     v
 Stream (captures changes)
     |
     | SYSTEM$STREAM_HAS_DATA('stream') = TRUE
     v
 Task (fires on schedule, only when stream has data)
     |
     | Executes: INSERT/MERGE INTO target FROM stream
     v
 Target Table (incrementally updated)
```

### Full Working Example

```sql
-- 1. Source table
CREATE OR REPLACE TABLE raw_orders (
  order_id INT,
  customer_id INT,
  amount DECIMAL(10,2),
  status VARCHAR,
  updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- 2. Target table
CREATE OR REPLACE TABLE processed_orders (
  order_id INT,
  customer_id INT,
  amount DECIMAL(10,2),
  status VARCHAR,
  loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- 3. Stream on source
CREATE OR REPLACE STREAM orders_stream ON TABLE raw_orders;

-- 4. Task that consumes the stream
CREATE OR REPLACE TASK process_orders_task
  WAREHOUSE = 'COMPUTE_WH'
  SCHEDULE = '1 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('orders_stream')
AS
  INSERT INTO processed_orders (order_id, customer_id, amount, status)
  SELECT order_id, customer_id, amount, status
  FROM orders_stream
  WHERE METADATA$ACTION = 'INSERT';

-- 5. Resume the task
ALTER TASK process_orders_task RESUME;
```

---
## 5. MERGE Pattern (Handling Updates + Deletes)

For full CDC (not just inserts), use MERGE:

```sql
CREATE OR REPLACE TASK cdc_merge_task
  WAREHOUSE = 'COMPUTE_WH'
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('orders_stream')
AS
  MERGE INTO processed_orders t
  USING (
    SELECT *
    FROM orders_stream
    WHERE METADATA$ACTION = 'INSERT'  -- new values (inserts + update-new)
  ) s
  ON t.order_id = s.order_id
  WHEN MATCHED THEN
    UPDATE SET
      t.customer_id = s.customer_id,
      t.amount = s.amount,
      t.status = s.status
  WHEN NOT MATCHED THEN
    INSERT (order_id, customer_id, amount, status)
    VALUES (s.order_id, s.customer_id, s.amount, s.status);
```

For deletes, you'd add a separate DELETE statement or handle them in the MERGE logic by checking `METADATA$ACTION = 'DELETE' AND METADATA$ISUPDATE = FALSE`.

---
## 6. Monitoring

### Check Stream Status
```sql
SHOW STREAMS;
SELECT SYSTEM$STREAM_HAS_DATA('my_stream');  -- TRUE/FALSE
```

### Check Task History
```sql
-- Recent runs
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
  TASK_NAME => 'PROCESS_ORDERS_TASK',
  SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC;
```

### Task State
```sql
SHOW TASKS;
-- Look at the 'state' column: 'started' = running, 'suspended' = paused
```

---
## 7. Common Pitfalls

| Pitfall | Explanation | Fix |
|---|---|---|
| Forgot to RESUME task | Tasks start suspended | `ALTER TASK my_task RESUME;` |
| Stream goes stale | Retention period < consumption gap | Increase `DATA_RETENTION_TIME_IN_DAYS` on source |
| SELECT from stream without DML | Offset won't advance | Always consume via INSERT/MERGE/etc. |
| Resuming DAG in wrong order | Parent starts but children are still suspended | Resume children first, then root |
| Task runs but stream is empty | No WHEN clause | Add `WHEN SYSTEM$STREAM_HAS_DATA(...)` |
| Multiple consumers on one stream | Each DML advances offset — second consumer sees nothing | Create a separate stream per consumer |

---
## 8. Quick Reference Card

| Operation | SQL |
|---|---|
| Create stream | `CREATE STREAM s ON TABLE t;` |
| Create append-only stream | `CREATE STREAM s ON TABLE t APPEND_ONLY = TRUE;` |
| Read stream changes | `SELECT * FROM s;` |
| Check for data | `SELECT SYSTEM$STREAM_HAS_DATA('s');` |
| Create task | `CREATE TASK t WAREHOUSE='W' SCHEDULE='5 MINUTE' AS <sql>;` |
| Create triggered task | Add `WHEN SYSTEM$STREAM_HAS_DATA('s')` |
| Resume task | `ALTER TASK t RESUME;` |
| Suspend task | `ALTER TASK t SUSPEND;` |
| Create child task | `CREATE TASK child AFTER parent AS <sql>;` |
| View task history | `SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(...));` |
| Drop stream | `DROP STREAM s;` |
| Drop task | `DROP TASK t;` |